In [0]:
%run /Workspace/Users/vnvarkhede@gmail.com/databricks-genai-data-analyst-copilot/notebooks/08_integration/01_copilot_orchestrator.py

In [0]:
# ================================================================
# CELL 2 — IMPORTS
# ================================================================

import json
import inspect

print("=" * 70)
print("PHASE 17 — PRODUCTION RESPONSE")
print("=" * 70)

print()
print("Imports: PASS")

In [0]:
# ================================================================
# CELL 3 — DEPENDENCY CHECK
# ================================================================

required_functions = [
    "classify_question",
    "generate_sql_request",
    "validate_sql",
    "execute_sql",
    "generate_question_embedding",
    "retrieve_documents",
    "build_rag_context",
    "generate_rag_answer",
    "decompose_hybrid_question",
    "run_sql_route",
    "run_hybrid_route",
    "assemble_hybrid_answer",
    "ask_copilot"
]


print("=" * 70)
print("DEPENDENCY CHECK")
print("=" * 70)


failed_dependencies = []


for function_name in required_functions:

    available = callable(
        globals().get(function_name)
    )

    print(
        f"{'PASS' if available else 'FAIL'} - "
        f"{function_name}"
    )

    if not available:

        failed_dependencies.append(
            function_name
        )


print()
print(
    "Total dependencies:",
    len(required_functions)
)

print(
    "Failed dependencies:",
    len(failed_dependencies)
)


if failed_dependencies:

    raise RuntimeError(
        "Production response cannot continue. "
        "Missing functions: "
        + ", ".join(
            failed_dependencies
        )
    )


print()
print("Dependency check: PASS")

In [0]:
# ================================================================
# CELL 4 — VERIFY PRODUCTION RAG GENERATOR
# ================================================================

print("=" * 70)
print("RAG ANSWER GENERATOR VERIFICATION")
print("=" * 70)


rag_function = globals().get(
    "generate_rag_answer"
)


if not callable(
    rag_function
):

    raise RuntimeError(
        "generate_rag_answer is not loaded."
    )


rag_source = inspect.getsource(
    rag_function
)


print(
    "Function:",
    rag_function
)

print()
print(
    "Source file:",
    inspect.getsourcefile(
        rag_function
    )
)


if "dummy answer" in rag_source.lower():

    raise RuntimeError(
        "ERROR: Dummy generate_rag_answer() "
        "detected."
    )


print()
print(
    "Dummy implementation detected: NO"
)

print(
    "Production RAG generator: PASS"
)

In [0]:
# ================================================================
# CELL 5 — VERIFY ASK_COPILOT RAG BINDING
# ================================================================

print("=" * 70)
print("ASK_COPILOT → RAG FUNCTION CHECK")
print("=" * 70)


orchestrator_rag_function = (
    ask_copilot.__globals__.get(
        "generate_rag_answer"
    )
)


print(
    "ask_copilot RAG function:",
    orchestrator_rag_function
)


print()
print(
    "Source file:",
    inspect.getsourcefile(
        orchestrator_rag_function
    )
)


orchestrator_rag_source = inspect.getsource(
    orchestrator_rag_function
)


if "dummy answer" in (
    orchestrator_rag_source.lower()
):

    raise RuntimeError(
        "ask_copilot is still using "
        "the dummy generate_rag_answer()."
    )


same_function = (
    orchestrator_rag_function
    is generate_rag_answer
)


print()
print(
    "Same function:",
    same_function
)


if not same_function:

    raise RuntimeError(
        "ask_copilot is not using "
        "the loaded production RAG generator."
    )


print()
print(
    "ask_copilot RAG binding: PASS"
)

In [0]:
# ================================================================
# CELL 6 — PRODUCTION RESPONSE FORMATTER
# ================================================================

def format_copilot_response(response):
    """
    Convert the internal copilot response
    into a consistent production response.
    """

    if response is None:

        return {
            "success": False,
            "question": None,
            "route": None,
            "answer": None,
            "sql": None,
            "data": None,
            "sources": [],
            "error": "Copilot returned None.",
            "execution_time_ms": None
        }


    if not isinstance(
        response,
        dict
    ):

        return {
            "success": False,
            "question": None,
            "route": None,
            "answer": None,
            "sql": None,
            "data": None,
            "sources": [],
            "error": (
                "Invalid response type: "
                + type(response).__name__
            ),
            "execution_time_ms": None
        }


    return {
        "success": response.get(
            "success",
            False
        ),

        "question": response.get(
            "question"
        ),

        "route": response.get(
            "route"
        ),

        "answer": response.get(
            "answer"
        ),

        "sql": response.get(
            "sql"
        ),

        "data": response.get(
            "data"
        ),

        "sources": response.get(
            "sources",
            []
        ),

        "error": response.get(
            "error"
        ),

        "execution_time_ms": response.get(
            "execution_time_ms"
        )
    }


print(
    "format_copilot_response(): PASS"
)

In [0]:
# ================================================================
# CELL 7 — PRODUCTION TEST: SQL
# ================================================================

print("=" * 70)
print("TEST 1 — SQL ROUTE")
print("=" * 70)


sql_question = (
    "Which region generated the highest revenue?"
)


sql_raw_response = ask_copilot(
    sql_question
)


sql_response = format_copilot_response(
    sql_raw_response
)


print(
    json.dumps(
        sql_response,
        indent=2,
        default=str
    )
)


assert sql_response["success"] is True

assert sql_response["route"] == "sql"

assert sql_response["error"] is None


print()
print("SQL ROUTE: PASS")

In [0]:
# ================================================================
# CELL 8 — PRODUCTION TEST: RAG
# ================================================================

print("=" * 70)
print("TEST 2 — RAG ROUTE")
print("=" * 70)


rag_question = (
    "What is the discount policy?"
)


rag_raw_response = ask_copilot(
    rag_question
)


rag_response = format_copilot_response(
    rag_raw_response
)


print(
    json.dumps(
        rag_response,
        indent=2,
        default=str
    )
)


assert rag_response["success"] is True

assert rag_response["route"] == "rag"

assert rag_response["error"] is None


rag_answer_text = str(
    rag_response["answer"]
)


assert (
    "dummy answer"
    not in rag_answer_text.lower()
)


print()
print("RAG ROUTE: PASS")
print("Real grounded answer detected: YES")

In [0]:
# ================================================================
# CELL 9 — PRODUCTION TEST: HYBRID
# ================================================================

print("=" * 70)
print("TEST 3 — HYBRID ROUTE")
print("=" * 70)


hybrid_question = (
    "Which region generated the highest revenue "
    "and what discount policy applies there?"
)


hybrid_raw_response = ask_copilot(
    hybrid_question
)


hybrid_response = format_copilot_response(
    hybrid_raw_response
)


print(
    json.dumps(
        hybrid_response,
        indent=2,
        default=str
    )
)


assert hybrid_response["success"] is True

assert hybrid_response["route"] == "hybrid"

assert hybrid_response["error"] is None


hybrid_answer_text = str(
    hybrid_response["answer"]
)


assert (
    "dummy answer"
    not in hybrid_answer_text.lower()
)


print()
print("HYBRID ROUTE: PASS")
print("Real grounded RAG answer detected: YES")

In [0]:
# ================================================================
# CELL 10 — SQL RESULT
# ================================================================

print("=" * 70)
print("SQL RESULT")
print("=" * 70)


if (
    sql_response["success"]
    and sql_response["data"] is not None
):

    display(
        sql_response["data"]
    )

else:

    print(
        "No SQL data available."
    )

In [0]:
# ================================================================
# CELL 11 — RAG RESULT
# ================================================================

print("=" * 70)
print("RAG RESULT")
print("=" * 70)


print(
    json.dumps(
        rag_response["answer"],
        indent=2,
        default=str
    )
)


print()
print("Sources:")


for source in rag_response["sources"]:

    print(
        f"- {source.get('file_name')} "
        f"— {source.get('title')}"
    )

In [0]:
# ================================================================
# CELL 12 — HYBRID RESULT
# ================================================================

print("=" * 70)
print("HYBRID RESULT")
print("=" * 70)


hybrid_answer = hybrid_response[
    "answer"
]


print(
    "Highest revenue region:",
    hybrid_answer.get(
        "highest_revenue_region"
    )
)


print(
    "Highest revenue:",
    hybrid_answer.get(
        "highest_revenue"
    )
)


print()
print("Discount policy:")
print(
    hybrid_answer.get(
        "discount_policy"
    )
)


print()
print("Sources:")


for source in hybrid_response["sources"]:

    print(
        f"- {source.get('file_name')} "
        f"— {source.get('title')}"
    )

In [0]:
# ================================================================
# CELL 13 — FINAL PRODUCTION VALIDATION
# ================================================================

print("=" * 70)
print("PHASE 17 — PRODUCTION VALIDATION")
print("=" * 70)


production_tests = [
    (
        "SQL",
        sql_response
    ),

    (
        "RAG",
        rag_response
    ),

    (
        "HYBRID",
        hybrid_response
    )
]


successful_tests = 0


for route_name, result in production_tests:

    passed = (
        result.get("success") is True
        and result.get("route") is not None
        and result.get("error") is None
    )


    print(
        f"{'PASS' if passed else 'FAIL'} - "
        f"{route_name}"
    )


    if passed:

        successful_tests += 1


print()
print(
    "Total production tests:",
    len(production_tests)
)

print(
    "Successful tests:",
    successful_tests
)

print(
    "Failed tests:",
    len(production_tests)
    - successful_tests
)


if successful_tests == len(
    production_tests
):

    print()
    print(
        "PHASE 17 STATUS: PASS ✓"
    )

else:

    print()
    print(
        "PHASE 17 STATUS: FAIL ✗"
    )

    raise RuntimeError(
        "Production validation failed."
    )

Your 01_copilot_orchestrator.py is now correctly loading the real RAG implementation, so 03_production_response should simply load → verify → test → validate.

Phase 18 — Production Hardening & Final API Response

Objective

Convert the successful internal ask_copilot() response into a clean, stable production response.

The key improvement is this:

Current RAG output

"answer": {
    "success": true,
    "answer": "The discount policy..."
}

Target production output

"answer": "The discount policy..."

We will also preserve sources correctly.